In [ ]:
"""
10x10 grid with obstacles and A* algorithm to find the shortest path from start to end
0 stands for empty space, 1 stands for obstacle

Time complexity: O(
"""
import heapq

class Node:
    def __init__(self, val, pos):
        self.val = val
        self.pos = pos
        self.g_cost = float('inf')  # Cost from start to this node
        self.h_cost = 0             # Heuristic cost (estimated cost to goal)
        self.f_cost = self.g_cost + self.h_cost
        self.parent = None          # For path reconstruction
    
    def __lt__(self, other):
        # Less than, for priority queue comparison
        return self.f_cost < other.f_cost


def convert_2_node_map(grid_map):
    node_map = [[None for _ in range(10)] for _ in range(10)]
    for i in range(10):
        for j in range(10):
            node_map[i][j] = Node(grid_map[i][j], (i, j))
    return node_map


def calc_h_cost(pos, end):
    # Manhattan distance heuristic, L1 norm
    return abs(pos[0] - end[0]) + abs(pos[1] - end[1])


def get_neighbors(node_map, node):
    neighbors = []
    i, j = node.pos
    directions = [(0, 1), (1, 0), (0, -1), (-1, 0)]

    for di, dj in directions:
        ni, nj = i + di, j + dj

        # Should be within grid bounds
        if 0 <= ni < 10 and 0 <= nj < 10:
            # Should not be an obstacle
            if node_map[ni][nj].val == 0:
                neighbors.append(node_map[ni][nj])

    return neighbors


def reconstruct_path(end_node):
    path = []
    current = end_node
    while current:
        path.append(current.pos)
        current = current.parent
    return path[::-1]


def astar(grid_map, start, end):
    node_map = convert_2_node_map(grid_map)

    start_node = node_map[start[0]][start[1]]
    end_node = node_map[end[0]][end[1]]

    # Check if start or end is an obstacle
    if start_node.val == 1 or end_node.val == 1:
        return None

    start_node.g_cost = 0
    start_node.h_cost = calc_h_cost(start, end)
    start_node.f_cost = start_node.h_cost

    to_be_visited_next = []
    heapq.heappush(to_be_visited_next, start_node)
    visited = set()

    while to_be_visited_next:
        # Get node with lowest f_cost
        current = heapq.heappop(to_be_visited_next)

        # Reconstruct path if reached end
        if current.pos == end:
            return reconstruct_path(current)

        visited.add(current.pos)

        for neighbor in get_neighbors(node_map, current):
            if neighbor.pos in visited:
                continue

            # Calculate tentative g_cost (cost from start)
            tentative_g_cost = current.g_cost + 1

            # If this path to neighbor is better than any previous one
            if tentative_g_cost < neighbor.g_cost: # neighbor.g_cost is inf originally
                neighbor.parent = current
                neighbor.g_cost = tentative_g_cost
                neighbor.h_cost = calc_h_cost(neighbor.pos, end)
                neighbor.f_cost = neighbor.g_cost + neighbor.h_cost

                # Add neighbor to next visit if not already there
                if all(item.pos != neighbor.pos for item in to_be_visited_next):
                    heapq.heappush(to_be_visited_next, neighbor)

    # No path found
    return None

In [4]:
grid_map = [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
            [0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
            [0, 0, 0, 0, 1, 1, 1, 0, 0, 0],
            [0, 0, 1, 1, 1, 0, 1, 0, 0, 0],
            [0, 0, 0, 0, 0, 0, 1, 0, 0, 0],
            [0, 0, 0, 0, 0, 0, 1, 0, 0, 0],
            [0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
            [0, 0, 0, 1, 1, 1, 0, 0, 0, 0],
            [0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
            [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]

start = (0, 0)
end = (7, 7)

path = astar(grid_map, start, end)

if path:
    print(f"Path found with {len(path)} steps:")
    print(path)
    
    # Visualize the path on the grid
    visual_map = [['□' if cell == 0 else '■' for cell in row] for row in grid_map]
    for i, j in path:
        if (i, j) == start:
            visual_map[i][j] = 'S'
        elif (i, j) == end:
            visual_map[i][j] = 'E'
        else:
            visual_map[i][j] = '*'
            
    for row in visual_map:
        print(' '.join(row))
else:
    print("No path found!")


Path found with 15 steps:
[(0, 0), (0, 1), (0, 2), (0, 3), (0, 4), (0, 5), (1, 5), (1, 6), (1, 7), (2, 7), (3, 7), (4, 7), (5, 7), (6, 7), (7, 7)]
S * * * * * □ □ □ □
□ □ □ □ ■ * * * □ □
□ □ □ □ ■ ■ ■ * □ □
□ □ ■ ■ ■ □ ■ * □ □
□ □ □ □ □ □ ■ * □ □
□ □ □ □ □ □ ■ * □ □
□ □ □ □ ■ □ □ * □ □
□ □ □ ■ ■ ■ □ E □ □
□ □ □ □ ■ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □
